# Online Retail Data Cleaning & Transformation Pipeline
### Optimized for Power BI Dashboard Creation
**Dataset:** Online Retail II (Transactions from 01/12/2009 to 09/12/2011)  
**Author:** Data Engineering & Analytics  
**Output:** Production-ready Cleaned Dataset for Power BI Modeling

---

## 📌 Project Overview & Objectives
This notebook carries out end-to-end data cleansing, anomaly removal, categorical standardization, and dimensional feature engineering on the Online Retail II dataset (`online_retail_II.xlsx`).

The primary objectives are:
1. **Consolidate multi-year sheets** (`Year 2009-2010` and `Year 2010-2011`).
2. **Audit data quality**: uncover duplicates, null values, negative/zero prices, and warehouse shrinkage adjustments.
3. **Handle edge cases**:
   - Differentiate valid customer cancellations from internal warehouse stock adjustments.
   - Remove accounting adjustments (e.g. Bad Debt write-offs) and administrative test stock codes.
   - Resolve missing Customer IDs without losing valid store revenue.
4. **Engineer Power BI dimensional features**:
   - Full Date & Time intelligence hierarchy (Year, Quarter, Month, YearMonth, DayName, Hour).
   - Order Status (`Completed` vs `Cancelled`) and Cancellation flag.
   - Line Total (`TotalAmount`).
   - Customer segmentation type (`Registered` vs `Guest`).
   - Product/Item categorization (`Merchandise`, `Postage`, `Manual`, etc.).
5. **Export optimized CSV files** for Power BI semantic model creation.

## 1. Environment Setup & Library Imports

In [8]:
import pandas as pd
import numpy as np
import os
import time

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
print(f"Pandas version: {pd.__version__}")

Pandas version: 3.0.2


## 2. Data Ingestion & Consolidation
Load both sheets (`Year 2009-2010` and `Year 2010-2011`) from `online_retail_II.xlsx` and concatenate into a single master dataframe.

In [9]:
file_path = 'online_retail_II.xlsx'

print("Loading Year 2009-2010...")
df_2009 = pd.read_excel(file_path, sheet_name='Year 2009-2010')
print(f"2009-2010 shape: {df_2009.shape}")

print("Loading Year 2010-2011...")
df_2010 = pd.read_excel(file_path, sheet_name='Year 2010-2011')
print(f"2010-2011 shape: {df_2010.shape}")

# Combine both years
df_raw = pd.concat([df_2009, df_2010], ignore_index=True)
print(f"\nTotal combined raw rows: {len(df_raw):,}")
df_raw.head()

Loading Year 2009-2010...
2009-2010 shape: (525461, 8)
Loading Year 2010-2011...
2010-2011 shape: (541910, 8)

Total combined raw rows: 1,067,371


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.00,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.00,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.00,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.00,United Kingdom


## 3. Exploratory Data Quality Audit
Analyze column types, missing values, duplicates, and statistical anomalies.

In [10]:
print("=== Data Info ===")
df_raw.info()

print("\n=== Missing Values Summary ===")
missing_df = pd.DataFrame({
    'Missing Count': df_raw.isnull().sum(),
    'Missing Percentage (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
display(missing_df)

print(f"\nExact duplicate rows: {df_raw.duplicated().sum():,}")

=== Data Info ===
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 78.8+ MB

=== Missing Values Summary ===


,Missing Count,Missing Percentage (%)
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00



Exact duplicate rows: 34,335


### 3.1 Distribution of Negative & Zero Values
Investigate negative quantities, zero prices, and negative prices.

In [11]:
print("Negative Quantities:", (df_raw['Quantity'] < 0).sum())
print("Zero Prices:", (df_raw['Price'] == 0).sum())
print("Negative Prices:", (df_raw['Price'] < 0).sum())

# Sample negative prices (Bad Debt adjustments)
display(df_raw[df_raw['Price'] < 0])

Negative Quantities: 22950
Zero Prices: 6202
Negative Prices: 5


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


## 4. Data Cleansing

### 4.1 Remove Exact Duplicates
Exact duplicates occur when identical transactions are logged multiple times during sync.

In [12]:
initial_rows = len(df_raw)
df_clean = df_raw.drop_duplicates().copy()
dropped_dups = initial_rows - len(df_clean)
print(f"Removed {dropped_dups:,} duplicate rows. Remaining rows: {len(df_clean):,}")

Removed 34,335 duplicate rows. Remaining rows: 1,033,036


### 4.2 Filter Non-Transaction Adjustments & Admin Codes
We remove:
1. **Negative prices**: internal accounting write-offs (`Adjust bad debt`).
2. **Zero prices**: warehouse shrinkage, damages, inventory checks without sales value.
3. **Administrative test stock codes**: `TEST001`, `TEST002`, `TEST`, `AMAZONFEE`, `CRUK`, `BANK CHARGES`, and `B`.

In [7]:
# Standardize StockCode and Invoice strings for filtering
df_clean['Invoice'] = df_clean['Invoice'].astype(str).str.strip()
df_clean['StockCode'] = df_clean['StockCode'].astype(str).str.strip().str.upper()

# Admin and test codes to exclude
admin_codes = ['TEST001', 'TEST002', 'TEST', 'CRUK', 'AMAZONFEE', 'BANK CHARGES', 'B']

# Apply validity mask: must have positive price and not be an admin/test code
valid_mask = (df_clean['Price'] > 0) & (~df_clean['StockCode'].isin(admin_codes))
excluded_records = (~valid_mask).sum()
print(f"Excluded non-transaction records: {excluded_records:,}")

df_clean = df_clean[valid_mask].copy()
print(f"Rows remaining after filtering: {len(df_clean):,}")

Excluded non-transaction records: 6,186
Rows remaining after filtering: 1,026,850


### 4.3 Differentiate Completed vs Cancelled Orders
Transactions with invoice starting with `'C'` or `Quantity < 0` represent customer returns / order cancellations.
In Power BI:
- Flagging rather than discarding cancellations enables analysis of **Gross Sales**, **Returns**, **Return Rates**, and **Net Sales**.
- We create `IsCancelled` (Boolean) and `OrderStatus` (`'Completed'` vs `'Cancelled'`).

In [13]:
df_clean['IsCancelled'] = df_clean['Invoice'].str.startswith('C') | (df_clean['Quantity'] < 0)
df_clean['OrderStatus'] = np.where(df_clean['IsCancelled'], 'Cancelled', 'Completed')

print("Order Status Distribution:")
print(df_clean['OrderStatus'].value_counts())

Order Status Distribution:
OrderStatus
Completed    1013932
Cancelled      19104
Name: count, dtype: int64


### 4.4 Standardize Customer ID & Handle Missing Values
In retail e-commerce, unauthenticated or guest transactions do not have a registered Customer ID.
- Dropping unauthenticated transactions would discard over £1.7M in revenue.
- Instead, we designate unregistered checkouts as `'Guest'` and create a `CustomerType` attribute (`'Registered'` vs `'Guest'`).
- Registered customer IDs are cleanly formatted without floating decimal points.

In [14]:
df_clean['CustomerID'] = df_clean['Customer ID'].apply(
    lambda x: f"{int(x)}" if pd.notnull(x) else 'Guest'
)
df_clean['CustomerType'] = np.where(df_clean['CustomerID'] == 'Guest', 'Guest', 'Registered')

print("Customer Type Breakdown:")
print(df_clean['CustomerType'].value_counts())

Customer Type Breakdown:
CustomerType
Registered    797885
Guest         235151
Name: count, dtype: int64


### 4.5 Text Cleaning & Country Normalization
- Strip extraneous whitespace and uppercase `Description`.
- Normalize country names (e.g. `'EIRE'` -> `'Ireland'`, `'RSA'` -> `'South Africa'`, `'USA'` -> `'United States'`).

In [15]:
# Clean description
df_clean['Description'] = df_clean['Description'].fillna('UNKNOWN').astype(str).str.strip().str.upper()

# Clean and normalize country
df_clean['Country'] = df_clean['Country'].astype(str).str.strip()
country_map = {
    'EIRE': 'Ireland',
    'RSA': 'South Africa',
    'USA': 'United States'
}
df_clean['Country'] = df_clean['Country'].replace(country_map)

print(f"Cleaned Country count: {df_clean['Country'].nunique()}")

Cleaned Country count: 43


## 5. Feature Engineering for Power BI

### 5.1 Financial Calculations & Item Categorization
- Calculate `TotalAmount = Quantity * Price` rounded to 2 decimal places.
- Classify `ItemCategory` (`Merchandise`, `Postage`, `Manual`, `Discount`, `Carriage`, `Accessories`).

In [16]:
# Financial line total
df_clean['TotalAmount'] = (df_clean['Quantity'] * df_clean['Price']).round(2)

# Item Category
def categorize_item(code):
    if code in ['POST', 'DOT']:
        return 'Postage'
    elif code == 'M':
        return 'Manual'
    elif code == 'D':
        return 'Discount'
    elif code == 'C2':
        return 'Carriage'
    elif code == 'PADS':
        return 'Accessories'
    else:
        return 'Merchandise'

df_clean['ItemCategory'] = df_clean['StockCode'].apply(categorize_item)
print("Item Category Distribution:")
print(df_clean['ItemCategory'].value_counts())

Item Category Distribution:
ItemCategory
Merchandise    1027669
Postage           3511
Manual            1387
Carriage           277
Discount           173
Accessories         19
Name: count, dtype: int64


### 5.2 Date & Time Intelligence Fields
Power BI dashboards require rich time hierarchies for slicing, dicing, and time-intelligence DAX functions:
- `Date` (YYYY-MM-DD)
- `Time` (HH:MM:SS)
- `Year`, `Quarter`, `Month`, `MonthName`, `YearMonth`
- `DayOfWeek`, `DayName`
- `Hour` (for hourly sales peak analysis)

In [17]:
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

df_clean['Date'] = df_clean['InvoiceDate'].dt.date
df_clean['Time'] = df_clean['InvoiceDate'].dt.strftime('%H:%M:%S')
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Quarter'] = 'Q' + df_clean['InvoiceDate'].dt.quarter.astype(str)
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['MonthName'] = df_clean['InvoiceDate'].dt.strftime('%B')
df_clean['YearMonth'] = df_clean['InvoiceDate'].dt.strftime('%Y-%m')
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.dayofweek + 1 # 1 = Monday, 7 = Sunday
df_clean['DayName'] = df_clean['InvoiceDate'].dt.strftime('%A')
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour

# Column organization
final_columns = [
    'Invoice', 'StockCode', 'Description', 'ItemCategory',
    'Quantity', 'Price', 'TotalAmount',
    'CustomerID', 'CustomerType', 'Country',
    'InvoiceDate', 'Date', 'Time', 'Year', 'Quarter', 'Month', 'MonthName', 'YearMonth', 'DayName', 'Hour',
    'IsCancelled', 'OrderStatus'
]

df_final = df_clean[final_columns].copy()
print(f"Final dataset shape: {df_final.shape}")
df_final.head(5)

Final dataset shape: (1033036, 22)


,Invoice,StockCode,Description,ItemCategory,Quantity,Price,TotalAmount,CustomerID,CustomerType,Country,InvoiceDate,Date,Time,Year,Quarter,Month,MonthName,YearMonth,DayName,Hour,IsCancelled,OrderStatus
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,Merchandise,12,6.95,83.40,13085,Registered,United Kingdom,2009-12-01 07:45:00,2009-12-01,07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,False,Completed
1,489434,79323P,PINK CHERRY LIGHTS,Merchandise,12,6.75,81.00,13085,Registered,United Kingdom,2009-12-01 07:45:00,2009-12-01,07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,False,Completed
2,489434,79323W,WHITE CHERRY LIGHTS,Merchandise,12,6.75,81.00,13085,Registered,United Kingdom,2009-12-01 07:45:00,2009-12-01,07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,False,Completed
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",Merchandise,48,2.10,100.80,13085,Registered,United Kingdom,2009-12-01 07:45:00,2009-12-01,07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,False,Completed
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,Merchandise,24,1.25,30.00,13085,Registered,United Kingdom,2009-12-01 07:45:00,2009-12-01,07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,False,Completed


## 6. Data Validation & Business Metrics Summary

In [18]:
gross_sales = df_final[df_final['OrderStatus'] == 'Completed']['TotalAmount'].sum()
refunds = df_final[df_final['OrderStatus'] == 'Cancelled']['TotalAmount'].sum()
net_sales = df_final['TotalAmount'].sum()

print("==================================================")
print("            ONLINE RETAIL METRICS SUMMARY         ")
print("==================================================")
print(f"Total Transactions / Records : {len(df_final):,}")
print(f"Completed Sales Count        : {(df_final['OrderStatus'] == 'Completed').sum():,}")
print(f"Cancelled / Returns Count    : {(df_final['OrderStatus'] == 'Cancelled').sum():,}")
print(f"Total Gross Sales            : £{gross_sales:,.2f}")
print(f"Total Returns / Refunds      : £{refunds:,.2f}")
print(f"Total Net Sales              : £{net_sales:,.2f}")
print(f"Unique Registered Customers  : {df_final[df_final['CustomerType'] == 'Registered']['CustomerID'].nunique():,}")
print(f"Unique Products (SKUs)       : {df_final[df_final['ItemCategory'] == 'Merchandise']['StockCode'].nunique():,}")
print(f"Countries Covered            : {df_final['Country'].nunique()}")
print(f"Date Range                   : {df_final['InvoiceDate'].min()} to {df_final['InvoiceDate'].max()}")
print("==================================================")

            ONLINE RETAIL METRICS SUMMARY         
Total Transactions / Records : 1,033,036
Completed Sales Count        : 1,013,932
Cancelled / Returns Count    : 19,104
Total Gross Sales            : £20,317,584.29
Total Returns / Refunds      : £-1,462,050.61
Total Net Sales              : £18,855,533.68
Unique Registered Customers  : 5,942
Unique Products (SKUs)       : 5,299
Countries Covered            : 43
Date Range                   : 2009-12-01 07:45:00 to 2011-12-09 12:50:00


## 7. Exporting Cleaned CSV Datasets for Power BI

1. **`online_retail_cleaned.csv`**: Full cleaned master dataset with order cancellation flags and enriched dimensions.

In [ ]:
output_clean = 'online_retail_cleaned.csv'

print(f"Exporting master cleaned dataset to '{output_clean}'...")
df_final.to_csv(output_clean, index=False, encoding='utf-8')
print(f"Export completed: {output_clean} ({os.path.getsize(output_clean) / (1024*1024):.2f} MB)")

print("\n[SUCCESS] Datasets are fully cleaned, validated, and ready for Power BI import!")

Exporting master cleaned dataset to 'online_retail_cleaned.csv'...
Export completed: online_retail_cleaned.csv (186.21 MB)

Exporting completed sales dataset to 'online_retail_completed_sales.csv'...
Export completed: online_retail_completed_sales.csv (182.74 MB)

[SUCCESS] Datasets are fully cleaned, validated, and ready for Power BI import!
